<a href="https://colab.research.google.com/github/Rhuan-Messias/T-picosIA/blob/main/v3_ml_cat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image
import numpy as np
import pandas as pd
import os
import shutil
from PIL import Image
from IPython.display import display, clear_output

In [ ]:
model = ResNet50(weights='imagenet')
historico_resultados = []

In [ ]:
if os.path.exists("uploads"): shutil.rmtree("uploads")
os.makedirs("uploads", exist_ok=True)

In [ ]:
print("⏳ Baixando e preparando dataset (100 imagens)...")
dataset_raw, info = tfds.load('cats_vs_dogs', split='train[:100]', with_info=True, as_supervised=True)

for i, (img_tensor, label) in enumerate(dataset_raw):
    # Definir nome baseado no label (1=gato, 0=cachorro)
    prefixo = "gato" if label == 1 else "nao_gato"
    filename = f"{prefixo}_{i}.jpg"
    img_path = os.path.join("uploads", filename)

    # Salvar tensor como arquivo JPG
    img_np = img_tensor.numpy().astype(np.uint8)
    Image.fromarray(img_np).save(img_path)

print("✅ Setup concluído! 100 imagens prontas na pasta 'uploads'.")

In [ ]:
def load_and_preprocess_image(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    # .copy() evita o erro de array somente leitura
    img_array = np.expand_dims(img_array, axis=0).copy()
    return preprocess_input(img_array)

def extract_label(filename):
    return "gato" if filename.lower().startswith("gato") else "nao_gato"

In [ ]:
cat_labels = ['tabby', 'tiger_cat', 'persian_cat', 'siamese_cat', 'egyptian_cat', 'cat', 'domestic_cat']

In [ ]:
print("🔍 Classificando imagens...")
arquivos = [f for f in os.listdir("uploads") if f.endswith(".jpg")]

for filename in arquivos:
    img_path = os.path.join("uploads", filename)
    try:
        img_array = load_and_preprocess_image(img_path)
        preds = model.predict(img_array, verbose=0)
        decoded = decode_predictions(preds, top=1)[0][0]

        label_imagenet = decoded[1]
        score = decoded[2]

        pred_gato = "gato" if any(cat in label_imagenet.lower() for cat in cat_labels) else "nao_gato"
        true_label = extract_label(filename)

        # Matriz de Confusão
        if true_label == "gato":
            res = "TP" if pred_gato == "gato" else "FN"
        else:
            res = "TN" if pred_gato == "nao_gato" else "FP"

        historico_resultados.append({
            "Imagem": filename,
            "Rótulo Real": true_label,
            "Predição": pred_gato,
            "Resultado": res
        })
    except Exception as e:
        print(f"❌ Erro em {filename}: {e}")

clear_output()
print(f"✅ Processamento finalizado! {len(historico_resultados)} imagens analisadas.")

In [ ]:
if not historico_resultados:
    print("A tabela está vazia.")
else:
    df = pd.DataFrame(historico_resultados)
    display(df)
    print(f"\n📊 Total de análises: {len(historico_resultados)}")

In [ ]:
import matplotlib.pyplot as plt
import math

def visualizar_galeria(historico, imagens_por_linha=8):
    if not historico: return

    total = len(historico[:24]) # Mostra apenas as primeiras 24 para não travar
    colunas = imagens_por_linha
    linhas = math.ceil(total / colunas)

    fig, axes = plt.subplots(linhas, colunas, figsize=(2.5 * colunas, 2.5 * linhas))
    axes = axes.flatten()

    for i in range(len(axes)):
        if i < total:
            item = historico[i]
            img_path = os.path.join("uploads", item['Imagem'])
            img = Image.open(img_path)
            axes[i].imshow(img)
            cor = 'red' if item['Resultado'] in ['FP', 'FN'] else 'green'
            axes[i].set_title(f"R:{item['Rótulo Real']}\nP:{item['Predição']}", color=cor, fontsize=8)
            axes[i].axis('off')
        else:
            axes[i].axis('off')
    plt.tight_layout()
    plt.show()

visualizar_galeria(historico_resultados)

In [ ]:
import shutil
import os
from IPython.display import clear_output

# 1. Esvazia a lista de dados
historico_resultados = []

# 2. Limpa a pasta física de imagens
if os.path.exists("uploads"):
    shutil.rmtree("uploads")
    os.makedirs("uploads")

# 3. Limpa a tela
clear_output()

print("🧹 SISTEMA RESETADO!")
print("Histórico limpo e pasta 'uploads' esvaziada.")
print("Para recomeçar, execute a Célula 1 novamente.")